# Dev Server

Start the document testing server for interactive OCR/NER/pipeline testing.

**Prerequisites**:
```
pip install flask flask-cors httpx
```

In [ ]:
import sys
from pathlib import Path

PROJECT = Path(".").resolve()
if not (PROJECT / "synthdoc").exists():
    PROJECT = PROJECT.parent

sys.path.insert(0, str(PROJECT))

## Configuration

In [ ]:
HOST = "0.0.0.0"
PORT = 5000
DEBUG = True
DEVICE = "cuda"          # cuda | xpu | cpu
INFERENCE = "vllm"       # local | api | vllm
VLLM_URL = "http://localhost:8000"
OPEN_BROWSER = True

## Start Server

Runs in a background thread. The web UI will be available at `http://<HOST>:<PORT>`.

In [ ]:
import os
import threading

os.environ["DEV_SERVER_DEVICE"] = DEVICE
os.environ["DEV_SERVER_INFERENCE"] = INFERENCE
os.environ["VLLM_URL"] = VLLM_URL

from dev_server.app import create_app

app = create_app()

if OPEN_BROWSER:
    import webbrowser
    threading.Timer(1.5, lambda: webbrowser.open(f"http://{HOST}:{PORT}")).start()

print(f"Dev server starting at http://{HOST}:{PORT}")
print(f"Device: {DEVICE} | Inference: {INFERENCE}")
if INFERENCE == "vllm":
    print(f"vLLM URL: {VLLM_URL}")

app.run(host=HOST, port=PORT, debug=DEBUG, use_reloader=False)

## Quick API Tests

Use these cells to test the API without the browser UI.

In [ ]:
import httpx

base = f"http://{HOST}:{PORT}"

r = httpx.get(f"{base}/health", timeout=5)
print("Health:", r.json())

In [ ]:
r = httpx.get(f"{base}/api/templates", timeout=10)
templates = r.json().get("templates", [])
print(f"{len(templates)} templates:")
for t in templates[:10]:
    print(f"  {t['name']}")
if len(templates) > 10:
    print(f"  ... and {len(templates) - 10} more")

In [ ]:
r = httpx.get(f"{base}/api/preview?template=invoice", timeout=30)
doc = r.json()
print(f"Template: {doc.get('template')}")
print(f"Size: {doc.get('width')}x{doc.get('height')}")
print(f"Annotations: {len(doc.get('annotations', []))}")
for a in doc.get("annotations", [])[:5]:
    print(f"  {a['entity_type']:15} {a.get('text_content', '')[:40]}")

In [ ]:
r = httpx.post(f"{base}/api/pipeline/run", json={
    "template": "invoice",
    "run_ocr": True,
    "run_ner": True,
    "compare_gt": True,
    "ocr_model": "qwen3-vl-8b",
    "ner_model": "qwen3-8b",
}, timeout=120)

result = r.json()

if result.get("success"):
    ocr = result.get("ocr", {})
    ner = result.get("ner", {})
    comp = result.get("comparison", {})

    print(f"OCR: {'OK' if ocr.get('success') else 'FAIL'}")
    if ocr.get('success'):
        print(f"  Text length: {len(ocr.get('text', ''))}")
        print(f"  Latency: {ocr.get('latency_ms', 0)}ms")

    print(f"\nNER: {'OK' if ner.get('success') else 'FAIL'}")
    if ner.get('success'):
        print(f"  Entities: {ner.get('count', 0)}")
        print(f"  Latency: {ner.get('latency_ms', 0)}ms")
        for e in ner.get('entities', [])[:5]:
            print(f"    {e.get('type', '?'):10} {e.get('text', '')[:40]}")

    print(f"\nComparison:")
    print(f"  P/R/F1: {comp.get('metrics', {})}")
else:
    print(f"Error: {result.get('error')}")